# OkunNLP — MT Fine-tuning Notebook
**Fine-tunes M2M100 and NLLB-200 on the English–Okun parallel corpus.**

### Before running:
- Runtime → Change runtime type → **T4 GPU**
- Run cells **in order, top to bottom**
- Total time: ~45 minutes
- The final cell prints all BLEU scores to copy into your paper

## Step 1 — Keep-alive (run first, before anything else)

In [ ]:
!pip install -q peft==0.10.0 --upgrade

In [ ]:
# Prevents Colab from disconnecting during long training runs
# Run this cell immediately after opening the notebook
import IPython
js = '''
function ClickConnect(){
    console.log('Keeping session alive...');
    document.querySelector('#top-toolbar > colab-connect-button')
        .shadowRoot.querySelector('#connect').click();
}
setInterval(ClickConnect, 60000);
'''
IPython.display.display(IPython.display.Javascript(js))
print('Keep-alive active.')

## Step 2 — Check GPU

In [ ]:
# Step 0: Fix peft/transformers version conflict
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers==4.40.2",
    "peft==0.10.0",
    "accelerate>=0.27.0"
], check=True)

print("Done. Now go to Runtime > Restart Runtime, then skip this cell and run from Step 1.")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU, then reconnect.')

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU : {gpu}')
print(f'VRAM: {vram:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

## Step 3 — Install dependencies

In [ ]:
%%capture
!pip install -q transformers==4.38.2 datasets sacrebleu sentencepiece \
    "accelerate==0.27.2" "peft==0.9.0" openpyxl pandas tqdm protobuf
print('Dependencies installed.')

## Step 4 — Mount Drive and load corpus

In [ ]:
import os
from google.colab import drive

# Check if the mountpoint exists and is not empty
mount_path = '/content/drive'
if os.path.exists(mount_path) and os.listdir(mount_path):
    print(f'Cleaning up non-empty mountpoint: {mount_path}')
    # Attempt to unmount and remove the directory to ensure a clean state
    try:
        drive.flush_and_unmount()
    except:
        pass
    !rm -rf "/content/drive"

drive.mount('/content/drive', force_remount=True)

In [ ]:
import pandas as pd, os

# ── UPDATE THIS PATH ──────────────────────────────────────────────────────
# To fix the OSError, download your Google Sheet as an Excel file (.xlsx) and update this path.
CORPUS_PATH = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/Okun Dataset Submission/finetune sample set.xlsx' # <<< UPDATE THIS TO YOUR .XLSX FILE
OUTPUT_DIR  = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/model_outputs'
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(CORPUS_PATH):
    raise FileNotFoundError(f'File not found: {CORPUS_PATH}\nUpdate CORPUS_PATH above.')

df = pd.read_excel(CORPUS_PATH)
en_col, ok_col = df.columns[1], df.columns[2]

# Keep only genuinely filled rows
filled_mask = (
    df[ok_col].notna() &
    (df[ok_col].astype(str).str.strip() != '') &
    (~df[ok_col].astype(str).str.contains(
        r'\[MISSING\]|\[UNVERIFIED\]|\[continues', na=False))
)
corpus = df[filled_mask][['Reference', en_col, ok_col]].reset_index(drop=True)
corpus.columns = ['reference', 'english', 'okun']

# Remove stray newlines inside cells
corpus['english'] = corpus['english'].str.replace(r'\s+', ' ', regex=True).str.strip()
corpus['okun']    = corpus['okun'].str.replace(r'\s+', ' ', regex=True).str.strip()

print(f'Loaded {len(corpus):,} sentence pairs')
print(corpus[['english','okun']].head(3).to_string())

## Step 5 — Split 80 / 10 / 10

In [ ]:
from datasets import Dataset

n = len(corpus)
t_end = int(n * 0.8)
d_end = int(n * 0.9)

train_df = corpus.iloc[:t_end].reset_index(drop=True)
dev_df   = corpus.iloc[t_end:d_end].reset_index(drop=True)
test_df  = corpus.iloc[d_end:].reset_index(drop=True)

train_ds = Dataset.from_pandas(train_df[['english','okun']])
dev_ds   = Dataset.from_pandas(dev_df[['english','okun']])
test_ds  = Dataset.from_pandas(test_df[['english','okun']])

print(f'Train : {len(train_ds):,}')
print(f'Dev   : {len(dev_ds):,}')
print(f'Test  : {len(test_ds):,}')

## Step 6 — Shared utilities (metrics + helpers)

In [ ]:
import time, gc, numpy as np
from sacrebleu.metrics import BLEU as SacreBLEU, CHRF

bleu_metric = SacreBLEU()
chrf_metric  = CHRF(word_order=2)   # word_order=2 → chrF++

def compute_metrics_fn(tokenizer):
    """Returns both BLEU and chrF++ so both are tracked during training."""
    def _fn(eval_preds):
        preds, labels = eval_preds
        # Replace -100 padding in labels before decoding
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds  = [p.strip() for p in decoded_preds]
        decoded_labels = [l.strip() for l in decoded_labels]
        bleu  = bleu_metric.corpus_score(decoded_preds, [decoded_labels]).score
        chrf  = chrf_metric.corpus_score(decoded_preds, [decoded_labels]).score
        return {'bleu': round(bleu, 4), 'chrf': round(chrf, 4)}
    return _fn

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()

def fmt_time(seconds):
    m, s = divmod(int(seconds), 60)
    return f'{m}m {s}s'

print('Utilities ready. Metrics: BLEU + chrF++ (word_order=2).')


In [ ]:
print(train_df[['english','okun']].head(3))
print(f"\nTotal rows: {len(train_df)}")
print(f"\nAny empty english: {train_df['english'].isna().sum()}")
print(f"Any empty okun: {train_df['okun'].isna().sum()}")

## Step 7 — Fine-tune M2M100
*Expected time: ~16 minutes on T4*

In [ ]:
!pip uninstall peft -y
!pip install peft==0.9.0 -q

In [ ]:
!pip install -q "accelerate==0.27.2" "peft==0.9.0"

In [ ]:
from transformers import (
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)

M2M_MODEL  = 'facebook/m2m100_418M'
M2M_OUTDIR = f'{OUTPUT_DIR}/m2m100_okun'
MAX_LEN    = 128

print('Loading M2M100 tokenizer and model...')
tok_m2m = M2M100Tokenizer.from_pretrained(M2M_MODEL, src_lang='en', tgt_lang='yo')
# ── Tokenise ──────────────────────────────────────────────────────────────
forced_bos = tok_m2m.get_lang_id('yo')
tok_m2m.src_lang = 'en'
tok_m2m.tgt_lang = 'yo'   # Yoruba — closest available proxy for Okun

model_m2m = M2M100ForConditionalGeneration.from_pretrained(
    M2M_MODEL,
    torch_dtype=torch.float32  # fp16
)
model_m2m.config.forced_bos_token_id = forced_bos



def preprocess_m2m(batch):
    tok_m2m.src_lang = 'en'
    model_inputs = tok_m2m(
        batch['english'],
        max_length=MAX_LEN, truncation=True, padding='max_length'
    )
    with tok_m2m.as_target_tokenizer():
        labels = tok_m2m(
            batch['okun'],
            max_length=MAX_LEN, truncation=True, padding='max_length'
        )
    label_ids = labels['input_ids']
    label_ids = [
        [(t if t != tok_m2m.pad_token_id else -100) for t in l]
        for l in label_ids
    ]
    model_inputs['labels'] = label_ids
    return model_inputs

print('Tokenising...')
tok_train_m2m = train_ds.map(preprocess_m2m, batched=True, remove_columns=train_ds.column_names)
tok_dev_m2m   = dev_ds.map(preprocess_m2m,   batched=True, remove_columns=dev_ds.column_names)
tok_test_m2m  = test_ds.map(preprocess_m2m,  batched=True, remove_columns=test_ds.column_names)

# ── Training args ─────────────────────────────────────────────────────────
args_m2m = Seq2SeqTrainingArguments(
    output_dir                  = M2M_OUTDIR,
    num_train_epochs            = 10,
    per_device_train_batch_size = 2,      # optimised for T4 with fp16
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 16,       # effective batch = 32
    learning_rate               = 5e-4,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    fp16                        = False,
    bf16                        = False,
    predict_with_generate       = True,
    generation_max_length       = MAX_LEN,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    greater_is_better           = True,
    save_total_limit            = 2,
    dataloader_num_workers      = 2,
    logging_steps               = 20,
    report_to                   = 'none',
)

collator_m2m = DataCollatorForSeq2Seq(tok_m2m, model=model_m2m, pad_to_multiple_of=8)

trainer_m2m = Seq2SeqTrainer(
    model           = model_m2m,
    args            = args_m2m,
    train_dataset   = tok_train_m2m,
    eval_dataset    = tok_dev_m2m,
    tokenizer       = tok_m2m,
    data_collator   = collator_m2m,
    compute_metrics = compute_metrics_fn(tok_m2m),
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print('\nStarting M2M100 training...')
t0 = time.time()
model_m2m = model_m2m.float()
trainer_m2m.train(resume_from_checkpoint=True)
m2m_train_time = time.time() - t0
print(f'\nM2M100 training done in {fmt_time(m2m_train_time)}')

# ── Evaluate & predict ────────────────────────────────────────────────────
print('Evaluating on dev set...')
eval_m2m = trainer_m2m.evaluate(eval_dataset=tok_dev_m2m)

print('Predicting on test set...')
pred_m2m = trainer_m2m.predict(tok_test_m2m)

print('\nM2M100 done.')
free_memory()

In [ ]:
# Check what the tokenized training data looks like
sample = tok_train_m2m[0]
print("Input ids:", sample['input_ids'][:10])
print("Labels:", sample['labels'][:10])
print("Any valid labels:", any(l != -100 for l in sample['labels']))

In [ ]:
# Test compute_metrics manually
import numpy as np
sample_preds = tok_train_m2m[:4]['input_ids']
sample_labels = tok_train_m2m[:4]['labels']
sample_preds = np.array(sample_preds)
sample_labels = np.array(sample_labels)
result = compute_metrics_fn(tok_m2m)((sample_preds, sample_labels))
print(result)

## Step 8 — Fine-tune NLLB-200
*Expected time: ~20 minutes on T4*

In [ ]:
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
MAX_LEN = 64

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from transformers import AutoModelForSeq2SeqLM, NllbTokenizerFast

NLLB_MODEL  = 'facebook/nllb-200-distilled-600M'
NLLB_OUTDIR = f'{OUTPUT_DIR}/nllb_okun'
NLLB_SRC    = 'eng_Latn'
NLLB_TGT    = 'yor_Latn'   # Yoruba — closest available proxy for Okun

print('Loading NLLB-200 tokenizer and model...')
tok_nllb = NllbTokenizerFast.from_pretrained(
    NLLB_MODEL, src_lang=NLLB_SRC, tgt_lang=NLLB_TGT
)
forced_bos_nllb = tok_nllb.convert_tokens_to_ids(NLLB_TGT)

model_nllb = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL,
    torch_dtype=torch.float32
)
model_nllb.config.forced_bos_token_id = forced_bos_nllb



def preprocess_nllb(batch):
    tok_nllb.src_lang = NLLB_SRC
    model_inputs = tok_nllb(
        batch['english'],
        max_length=MAX_LEN, truncation=True, padding='max_length'
    )
    with tok_nllb.as_target_tokenizer():
        labels = tok_nllb(
            batch['okun'],
            max_length=MAX_LEN, truncation=True, padding='max_length'
        )
    label_ids = labels['input_ids']
    label_ids = [
        [(t if t != tok_nllb.pad_token_id else -100) for t in l]
        for l in label_ids
    ]
    model_inputs['labels'] = label_ids
    return model_inputs

print('Tokenising...')
tok_train_nllb = train_ds.map(preprocess_nllb, batched=True, remove_columns=train_ds.column_names)
tok_dev_nllb   = dev_ds.map(preprocess_nllb,   batched=True, remove_columns=dev_ds.column_names)
tok_test_nllb  = test_ds.map(preprocess_nllb,  batched=True, remove_columns=test_ds.column_names)

args_nllb = Seq2SeqTrainingArguments(
    output_dir                  = NLLB_OUTDIR,
    num_train_epochs            = 10,
    per_device_train_batch_size = 1,       # NLLB is larger; reduce batch size
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = 32,       # effective batch = 32
    learning_rate               = 5e-4,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    fp16                        = False,
    bf16                        = False,
    predict_with_generate       = True,
    generation_max_length       = MAX_LEN,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    greater_is_better           = True,
    save_total_limit            = 2,
    dataloader_num_workers      = 0,
    logging_steps               = 20,
    report_to                   = 'none',
)

collator_nllb = DataCollatorForSeq2Seq(tok_nllb, model=model_nllb, pad_to_multiple_of=8)

trainer_nllb = Seq2SeqTrainer(
    model           = model_nllb,
    args            = args_nllb,
    train_dataset   = tok_train_nllb,
    eval_dataset    = tok_dev_nllb,
    tokenizer       = tok_nllb,
    data_collator   = collator_nllb,
    compute_metrics = compute_metrics_fn(tok_nllb),
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

model_nllb = model_nllb.float()
model_nllb.config.use_cache = False
print('\nStarting NLLB-200 training...')
t0 = time.time()
trainer_nllb.train(resume_from_checkpoint=True)
nllb_train_time = time.time() - t0
print(f'\nNLLB-200 training done in {fmt_time(nllb_train_time)}')

print('Evaluating on dev set...')
eval_nllb = trainer_nllb.evaluate(eval_dataset=tok_dev_nllb)

print('Predicting on test set...')
pred_nllb = trainer_nllb.predict(tok_test_nllb)

print('\nNLLB-200 done.')
free_memory()

In [ ]:
import os, json

def inspect_checkpoints(model_dir, model_name):
    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")

    checkpoints = sorted([
        d for d in os.listdir(model_dir)
        if d.startswith('checkpoint-')
    ], key=lambda x: int(x.split('-')[1]))

    for ckpt in checkpoints:
        ckpt_path = os.path.join(model_dir, ckpt)
        step = ckpt.split('-')[1]

        # Try to read trainer state for BLEU and loss
        trainer_state = os.path.join(ckpt_path, 'trainer_state.json')
        if os.path.exists(trainer_state):
            with open(trainer_state) as f:
                state = json.load(f)
            best_ckpt = state.get('best_model_checkpoint', 'unknown')
            best_metric = state.get('best_metric', 'unknown')
            # Find this checkpoint's eval log
            logs = state.get('log_history', [])
            eval_logs = [l for l in logs if 'eval_bleu' in l and
                        l.get('step') == int(step)]
            bleu = eval_logs[0].get('eval_bleu', 'N/A') if eval_logs else 'N/A'
            loss = eval_logs[0].get('eval_loss', 'N/A') if eval_logs else 'N/A'

            print(f"\n  Checkpoint: {ckpt}")
            print(f"    Eval BLEU : {bleu}")
            print(f"    Eval Loss : {loss}")
            print(f"    Best ckpt : {os.path.basename(best_ckpt)}")
            print(f"    Best metric (overall): {best_metric}")
        else:
            print(f"\n  Checkpoint: {ckpt} — no trainer_state.json found")

inspect_checkpoints(f'{OUTPUT_DIR}/m2m100_okun', 'M2M100')
inspect_checkpoints(f'{OUTPUT_DIR}/nllb_okun',   'NLLB-200')

In [ ]:
import os
os.remove(f"{OUTPUT_DIR}/m2m_eval.json")

#gpu version

In [ ]:
# ── 0. Install packages ───────────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q', 'evaluate', 'sacrebleu'], check=True)

# ── 1. Imports ────────────────────────────────────────────────────────────
import time, torch, numpy as np, os, json, gc
import pandas as pd
import evaluate as hf_evaluate
from datasets import Dataset
from transformers import (
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    AutoModelForSeq2SeqLM, NllbTokenizerFast,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from sklearn.model_selection import train_test_split

# ── 2. Use GPU ────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── 3. Paths ──────────────────────────────────────────────────────────────
CORPUS_PATH  = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/Okun Dataset Submission/finetune sample set.xlsx'
OUTPUT_DIR   = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/model_outputs'
M2M_CKPT     = f'{OUTPUT_DIR}/m2m100_okun/checkpoint-392'
NLLB_CKPT    = f'{OUTPUT_DIR}/nllb_okun/checkpoint-440'

os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LEN   = 128
NLLB_SRC  = 'eng_Latn'
NLLB_TGT  = 'yor_Latn'

# ⚡ Debug mode
QUICK_TEST = False
SAMPLE_SIZE = 50

# ── 4. Load corpus ────────────────────────────────────────────────────────
print("\nLoading corpus...")
df = pd.read_excel(CORPUS_PATH)
en_col, ok_col = df.columns[1], df.columns[2]

filled_mask = (
    df[ok_col].notna() &
    (df[ok_col].astype(str).str.strip() != '') &
    (~df[ok_col].astype(str).str.contains(r'\[MISSING\]|\[UNVERIFIED\]|\[continues', na=False))
)

corpus = df[filled_mask][['Reference', en_col, ok_col]].reset_index(drop=True)
corpus.columns = ['reference', 'english', 'okun']

corpus['english'] = corpus['english'].str.replace(r'\s+', ' ', regex=True).str.strip()
corpus['okun']    = corpus['okun'].str.replace(r'\s+', ' ', regex=True).str.strip()

print(f'Loaded {len(corpus):,} sentence pairs')

train_df, temp_df = train_test_split(corpus, test_size=0.2, random_state=42)
dev_df,  test_df  = train_test_split(temp_df, test_size=0.5, random_state=42)

dev_ds  = Dataset.from_pandas(dev_df[['english','okun']].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['english','okun']].reset_index(drop=True))

if QUICK_TEST:
    dev_ds  = dev_ds.select(range(SAMPLE_SIZE))
    test_ds = test_ds.select(range(SAMPLE_SIZE))
    print(f"⚡ Quick test mode ON ({SAMPLE_SIZE} samples)")

# ── 5. Metrics ────────────────────────────────────────────────────────────
bleu_metric = hf_evaluate.load('sacrebleu')

def compute_metrics_fn(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]

        decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        decoded_preds  = [p.strip() for p in decoded_preds]
        decoded_labels = [[l.strip()] for l in decoded_labels]

        result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
        return {'bleu': round(result['score'], 4)}

    return compute_metrics

def file_valid(path):
    return os.path.exists(path) and os.path.getsize(path) > 0

# ── 6. M2M100 ─────────────────────────────────────────────────────────────
if file_valid(f"{OUTPUT_DIR}/m2m_eval.json"):
    print("\n Skipping M2M (already done)")
else:
    print("\nLoading M2M100...")

    tok_m2m = M2M100Tokenizer.from_pretrained(M2M_CKPT, src_lang='en', tgt_lang='yo')
    model_m2m = M2M100ForConditionalGeneration.from_pretrained(
        M2M_CKPT, dtype=torch.float16
    ).to(device)



    def preprocess_m2m(batch):
        model_inputs = tok_m2m(batch['english'], max_length=MAX_LEN, truncation=True, padding='max_length')
        labels = tok_m2m(text_target=batch['okun'], max_length=MAX_LEN, truncation=True, padding='max_length')
        model_inputs['labels'] = [
            [(t if t != tok_m2m.pad_token_id else -100) for t in l]
            for l in labels['input_ids']
        ]
        return model_inputs

    tok_dev_m2m  = dev_ds.map(preprocess_m2m, batched=True, remove_columns=dev_ds.column_names)
    tok_test_m2m = test_ds.map(preprocess_m2m, batched=True, remove_columns=test_ds.column_names)

    args_m2m = Seq2SeqTrainingArguments(
        output_dir='/tmp/m2m_eval',
        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        generation_num_beams=1,
        per_device_eval_batch_size=4,   # GPU boost
        fp16=(device.type == 'cuda'),
        report_to='none',
    )

    trainer = Seq2SeqTrainer(
        model=model_m2m,
        args=args_m2m,
        processing_class=tok_m2m,
        data_collator=DataCollatorForSeq2Seq(tok_m2m, model=model_m2m),
        compute_metrics=compute_metrics_fn(tok_m2m),
    )

    print("Evaluating M2M...")
    eval_m2m = trainer.evaluate(tok_dev_m2m)

    print("Predicting M2M...")
    pred_m2m = trainer.predict(tok_test_m2m)

    # SAVE
    with open(f"{OUTPUT_DIR}/m2m_eval.json", "w") as f:
        json.dump(eval_m2m, f, indent=2)

    decoded_preds = tok_m2m.batch_decode(pred_m2m.predictions, skip_special_tokens=True)

    with open(f"{OUTPUT_DIR}/m2m_predictions.txt", "w") as f:
        f.write("\n".join(decoded_preds))

    print("✅ M2M saved")

    del model_m2m
    torch.cuda.empty_cache()
    gc.collect()

# ── 7. NLLB ───────────────────────────────────────────────────────────────
if file_valid(f"{OUTPUT_DIR}/nllb_eval.json"):
    print("\n Skipping NLLB (already done)")
else:
    print("\nLoading NLLB...")

    tok_nllb = NllbTokenizerFast.from_pretrained(
        'facebook/nllb-200-distilled-600M',
        src_lang=NLLB_SRC,
        tgt_lang=NLLB_TGT
    )

    model_nllb = AutoModelForSeq2SeqLM.from_pretrained(
        NLLB_CKPT,
        torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32
    ).to(device)

    model_nllb.generation_config.forced_bos_token_id = tok_nllb.convert_tokens_to_ids(NLLB_TGT)

    def preprocess_nllb(batch):
        model_inputs = tok_nllb(batch['english'], max_length=MAX_LEN, truncation=True, padding='max_length')
        labels = tok_nllb(text_target=batch['okun'], max_length=MAX_LEN, truncation=True, padding='max_length')
        model_inputs['labels'] = [
            [(t if t != tok_nllb.pad_token_id else -100) for t in l]
            for l in labels['input_ids']
        ]
        return model_inputs

    tok_dev_nllb  = dev_ds.map(preprocess_nllb, batched=True, remove_columns=dev_ds.column_names)
    tok_test_nllb = test_ds.map(preprocess_nllb, batched=True, remove_columns=test_ds.column_names)

    args_nllb = Seq2SeqTrainingArguments(
        output_dir='/tmp/nllb_eval',
        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        generation_num_beams=1,
        per_device_eval_batch_size=4,
        fp16=(device.type == 'cuda'),
        report_to='none',
    )

    trainer = Seq2SeqTrainer(
        model=model_nllb,
        args=args_nllb,
        processing_class=tok_nllb,
        data_collator=DataCollatorForSeq2Seq(tok_nllb, model=model_nllb),
        compute_metrics=compute_metrics_fn(tok_nllb),
    )

    print("Evaluating NLLB...")
    eval_nllb = trainer.evaluate(tok_dev_nllb)

    print("Predicting NLLB...")
    pred_nllb = trainer.predict(tok_test_nllb)

    # SAVE
    with open(f"{OUTPUT_DIR}/nllb_eval.json", "w") as f:
        json.dump(eval_nllb, f, indent=2)

    decoded_preds = tok_nllb.batch_decode(pred_nllb.predictions, skip_special_tokens=True)

    with open(f"{OUTPUT_DIR}/nllb_predictions.txt", "w") as f:
        f.write("\n".join(decoded_preds))

    print("✅ NLLB saved")

print("\n🎉 DONE — GPU, saved, restart-safe")

#CPU Version

In [ ]:
# ── 0. Install missing packages ───────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q', 'evaluate', 'sacrebleu'], check=True)

# ── 1. Imports ────────────────────────────────────────────────────────────
import time, torch, numpy as np, os, json, gc
import pandas as pd
import evaluate as hf_evaluate
from datasets import Dataset
from transformers import (
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    AutoModelForSeq2SeqLM, NllbTokenizerFast,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from sklearn.model_selection import train_test_split

# ── 2. Force CPU ──────────────────────────────────────────────────────────
device = torch.device('cpu')
print(f"Using device: {device}")

# ── 3. Paths ──────────────────────────────────────────────────────────────
CORPUS_PATH  = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/Okun Dataset Submission/finetune sample set.xlsx'
OUTPUT_DIR   = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/model_outputs'
M2M_CKPT     = f'{OUTPUT_DIR}/m2m100_okun/checkpoint-392'
NLLB_CKPT    = f'{OUTPUT_DIR}/nllb_okun/checkpoint-440'

os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LEN      = 128
NLLB_SRC     = 'eng_Latn'
NLLB_TGT     = 'yor_Latn'

#  SET THIS TO True FOR FAST DEBUG RUNS
QUICK_TEST = False
SAMPLE_SIZE = 50

# ── 4. Load corpus ────────────────────────────────────────────────────────
print("\nLoading corpus...")
df = pd.read_excel(CORPUS_PATH)
en_col, ok_col = df.columns[1], df.columns[2]

filled_mask = (
    df[ok_col].notna() &
    (df[ok_col].astype(str).str.strip() != '') &
    (~df[ok_col].astype(str).str.contains(r'\[MISSING\]|\[UNVERIFIED\]|\[continues', na=False))
)

corpus = df[filled_mask][['Reference', en_col, ok_col]].reset_index(drop=True)
corpus.columns = ['reference', 'english', 'okun']

corpus['english'] = corpus['english'].str.replace(r'\s+', ' ', regex=True).str.strip()
corpus['okun']    = corpus['okun'].str.replace(r'\s+', ' ', regex=True).str.strip()

print(f'Loaded {len(corpus):,} sentence pairs')

train_df, temp_df = train_test_split(corpus, test_size=0.2, random_state=42)
dev_df,  test_df  = train_test_split(temp_df, test_size=0.5, random_state=42)

dev_ds  = Dataset.from_pandas(dev_df[['english','okun']].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['english','okun']].reset_index(drop=True))

#  QUICK TEST MODE
if QUICK_TEST:
    dev_ds = dev_ds.select(range(SAMPLE_SIZE))
    test_ds = test_ds.select(range(SAMPLE_SIZE))
    print(f" Quick test mode ON ({SAMPLE_SIZE} samples)")

# ── 5. Metrics ────────────────────────────────────────────────────────────
bleu_metric = hf_evaluate.load('sacrebleu')

def compute_metrics_fn(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]

        decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        decoded_preds  = [p.strip() for p in decoded_preds]
        decoded_labels = [[l.strip()] for l in decoded_labels]

        result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
        return {'bleu': round(result['score'], 4)}

    return compute_metrics

# ── 6. M2M100 ─────────────────────────────────────────────────────────────
if os.path.exists(f"{OUTPUT_DIR}/m2m_eval.json"):
    print("\n Skipping M2M (already done)")
else:
    print("\nLoading M2M100...")
    tok_m2m = M2M100Tokenizer.from_pretrained(M2M_CKPT, src_lang='en', tgt_lang='yo')
    model_m2m = M2M100ForConditionalGeneration.from_pretrained(
        M2M_CKPT, dtype=torch.float32
    ).to(device)

    model_m2m = M2M100ForConditionalGeneration.from_pretrained(
        M2M_CKPT, dtype=torch.float16
    ).to(device)

    def preprocess_m2m(batch):
        model_inputs = tok_m2m(batch['english'], max_length=MAX_LEN, truncation=True, padding='max_length')
        labels = tok_m2m(text_target=batch['okun'], max_length=MAX_LEN, truncation=True, padding='max_length')
        model_inputs['labels'] = [
            [(t if t != tok_m2m.pad_token_id else -100) for t in l]
            for l in labels['input_ids']
        ]
        return model_inputs

    tok_dev_m2m  = dev_ds.map(preprocess_m2m, batched=True, remove_columns=dev_ds.column_names)
    tok_test_m2m = test_ds.map(preprocess_m2m, batched=True, remove_columns=test_ds.column_names)

    args_m2m = Seq2SeqTrainingArguments(
        output_dir='/tmp/m2m_eval',
        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        generation_num_beams=1,   # SPEED BOOST
        per_device_eval_batch_size=1,
        fp16=False,
        bf16=False,
        use_cpu=True,
        report_to='none',
    )

    trainer = Seq2SeqTrainer(
        model=model_m2m,
        args=args_m2m,
        processing_class=tok_m2m,
        data_collator=DataCollatorForSeq2Seq(tok_m2m, model=model_m2m),
        compute_metrics=compute_metrics_fn(tok_m2m),
    )

    print("Evaluating M2M...")
    eval_m2m = trainer.evaluate(tok_dev_m2m)

    print("Predicting M2M...")
    pred_m2m = trainer.predict(tok_test_m2m)

    # SAVE RESULTS
    with open(f"{OUTPUT_DIR}/m2m_eval.json", "w") as f:
        json.dump(eval_m2m, f, indent=2)

    decoded_preds = tok_m2m.batch_decode(pred_m2m.predictions, skip_special_tokens=True)

    with open(f"{OUTPUT_DIR}/m2m_predictions.txt", "w") as f:
        f.write("\n".join(decoded_preds))

    print("✅ M2M saved")

    del model_m2m
    gc.collect()

# ── 7. NLLB ───────────────────────────────────────────────────────────────
if os.path.exists(f"{OUTPUT_DIR}/nllb_eval.json"):
    print("\n Skipping NLLB (already done)")
else:
    print("\nLoading NLLB...")

    tok_nllb = NllbTokenizerFast.from_pretrained(
        'facebook/nllb-200-distilled-600M',
        src_lang=NLLB_SRC,
        tgt_lang=NLLB_TGT
    )

    model_nllb = AutoModelForSeq2SeqLM.from_pretrained(
        NLLB_CKPT, dtype=torch.float32
    ).to(device)

    # ✅ FIXED
    model_nllb.generation_config.forced_bos_token_id = tok_nllb.convert_tokens_to_ids(NLLB_TGT)

    def preprocess_nllb(batch):
        model_inputs = tok_nllb(batch['english'], max_length=MAX_LEN, truncation=True, padding='max_length')
        labels = tok_nllb(text_target=batch['okun'], max_length=MAX_LEN, truncation=True, padding='max_length')
        model_inputs['labels'] = [
            [(t if t != tok_nllb.pad_token_id else -100) for t in l]
            for l in labels['input_ids']
        ]
        return model_inputs

    tok_dev_nllb  = dev_ds.map(preprocess_nllb, batched=True, remove_columns=dev_ds.column_names)
    tok_test_nllb = test_ds.map(preprocess_nllb, batched=True, remove_columns=test_ds.column_names)

    args_nllb = Seq2SeqTrainingArguments(
        output_dir='/tmp/nllb_eval',
        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        generation_num_beams=1,   #  SPEED BOOST
        per_device_eval_batch_size=1,
        fp16=False,
        bf16=False,
        use_cpu=True,
        report_to='none',
    )

    trainer = Seq2SeqTrainer(
        model=model_nllb,
        args=args_nllb,
        processing_class=tok_nllb,
        data_collator=DataCollatorForSeq2Seq(tok_nllb, model=model_nllb),
        compute_metrics=compute_metrics_fn(tok_nllb),
    )

    print("Evaluating NLLB...")
    eval_nllb = trainer.evaluate(tok_dev_nllb)

    print("Predicting NLLB...")
    pred_nllb = trainer.predict(tok_test_nllb)

    # SAVE RESULTS
    with open(f"{OUTPUT_DIR}/nllb_eval.json", "w") as f:
        json.dump(eval_nllb, f, indent=2)

    decoded_preds = tok_nllb.batch_decode(pred_nllb.predictions, skip_special_tokens=True)

    with open(f"{OUTPUT_DIR}/nllb_predictions.txt", "w") as f:
        f.write("\n".join(decoded_preds))

    print("✅ NLLB saved")

print("\n🎉 DONE — safe, saved, and restartable")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_excel(CORPUS_PATH)

train_df, temp_df = train_test_split(corpus, test_size=0.2, random_state=42)
dev_df,  test_df  = train_test_split(temp_df, test_size=0.5, random_state=42)

# Check for reference overlap
train_refs = set(train_df['reference'])
test_refs  = set(test_df['reference'])
overlap = train_refs & test_refs
print(f"Reference overlap between train and test: {len(overlap)}")

# Check for English text overlap (near-duplicates)
train_en = set(train_df['english'].str.strip())
test_en  = set(test_df['english'].str.strip())
en_overlap = train_en & test_en
print(f"Exact English text overlap: {len(en_overlap)}")


In [ ]:
# @title
# ── 0. Install missing packages ───────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q', 'evaluate', 'sacrebleu'], check=True)

# ── 1. Imports ────────────────────────────────────────────────────────────
import time, torch, numpy as np, evaluate as hf_evaluate
from datasets import load_from_disk
from transformers import (
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    AutoModelForSeq2SeqLM, NllbTokenizerFast,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

# ── 2. Paths ──────────────────────────────────────────────────────────────
OUTPUT_DIR   = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/model_outputs'
M2M_CKPT     = f'{OUTPUT_DIR}/m2m100_okun/checkpoint-490'
NLLB_CKPT    = f'{OUTPUT_DIR}/nllb_okun/checkpoint-392'
MAX_LEN      = 128
NLLB_SRC     = 'eng_Latn'
NLLB_TGT     = 'yor_Latn'

# ── 3. Helper functions ───────────────────────────────────────────────────
def fmt_time(seconds):
    m, s = divmod(int(seconds), 60)
    return f"{m}m {s}s"

bleu_metric = hf_evaluate.load('sacrebleu')

def compute_metrics_fn(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds  = [p.strip() for p in decoded_preds]
        decoded_labels = [[l.strip()] for l in decoded_labels]
        result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
        return {'bleu': round(result['score'], 4)}
    return compute_metrics

# ── 4. Recover training times ─────────────────────────────────────────────
m2m_train_time  = 63 * 60 + 57
nllb_train_time = 90 * 60 + 0

# ── 5. Load your dataset splits ───────────────────────────────────────────
# Replace this block with however your notebook originally loads dev/test
# e.g. from a CSV or HuggingFace dataset on Drive
# If you saved them as arrow/HF datasets uncomment the lines below:
# dev_ds  = load_from_disk(f'{OUTPUT_DIR}/dev_ds')
# test_ds = load_from_disk(f'{OUTPUT_DIR}/test_ds')

# ── 6. Load M2M100 and tokenise ───────────────────────────────────────────
print("Loading M2M100 from checkpoint...")
tok_m2m = M2M100Tokenizer.from_pretrained(M2M_CKPT, src_lang='en', tgt_lang='yo')
model_m2m = M2M100ForConditionalGeneration.from_pretrained(M2M_CKPT, torch_dtype=torch.float32)
model_m2m.config.forced_bos_token_id = tok_m2m.get_lang_id('yo')

def preprocess_m2m(batch):
    tok_m2m.src_lang = 'en'
    model_inputs = tok_m2m(batch['english'], max_length=MAX_LEN, truncation=True, padding='max_length')
    labels = tok_m2m(text_target=batch['okun'], max_length=MAX_LEN, truncation=True, padding='max_length')
    model_inputs['labels'] = [
        [(t if t != tok_m2m.pad_token_id else -100) for t in l]
        for l in labels['input_ids']
    ]
    return model_inputs

print("Tokenising M2M100...")
tok_dev_m2m  = dev_ds.map(preprocess_m2m, batched=True, remove_columns=dev_ds.column_names)
tok_test_m2m = test_ds.map(preprocess_m2m, batched=True, remove_columns=test_ds.column_names)

args_m2m = Seq2SeqTrainingArguments(
    output_dir=M2M_CKPT, predict_with_generate=True,
    generation_max_length=MAX_LEN, per_device_eval_batch_size=2,
    fp16=False, bf16=False, report_to='none',
)
collator_m2m  = DataCollatorForSeq2Seq(tok_m2m, model=model_m2m, pad_to_multiple_of=8)
trainer_m2m_r = Seq2SeqTrainer(
    model=model_m2m, args=args_m2m,
    tokenizer=tok_m2m, data_collator=collator_m2m,
    compute_metrics=compute_metrics_fn(tok_m2m),
)
print("Evaluating M2M100...")
eval_m2m = trainer_m2m_r.evaluate(eval_dataset=tok_dev_m2m)
print("Predicting M2M100...")
pred_m2m = trainer_m2m_r.predict(tok_test_m2m)
print("M2M100 done:", eval_m2m)

# ── 7. Load NLLB and tokenise ─────────────────────────────────────────────
print("\nLoading NLLB from checkpoint...")
tok_nllb = NllbTokenizerFast.from_pretrained(NLLB_CKPT, src_lang=NLLB_SRC, tgt_lang=NLLB_TGT)
model_nllb = AutoModelForSeq2SeqLM.from_pretrained(NLLB_CKPT, torch_dtype=torch.float32)
model_nllb.config.forced_bos_token_id = tok_nllb.convert_tokens_to_ids(NLLB_TGT)

def preprocess_nllb(batch):
    tok_nllb.src_lang = NLLB_SRC
    model_inputs = tok_nllb(batch['english'], max_length=MAX_LEN, truncation=True, padding='max_length')
    labels = tok_nllb(text_target=batch['okun'], max_length=MAX_LEN, truncation=True, padding='max_length')
    model_inputs['labels'] = [
        [(t if t != tok_nllb.pad_token_id else -100) for t in l]
        for l in labels['input_ids']
    ]
    return model_inputs

print("Tokenising NLLB...")
tok_dev_nllb  = dev_ds.map(preprocess_nllb, batched=True, remove_columns=dev_ds.column_names)
tok_test_nllb = test_ds.map(preprocess_nllb, batched=True, remove_columns=test_ds.column_names)

args_nllb = Seq2SeqTrainingArguments(
    output_dir=NLLB_CKPT, predict_with_generate=True,
    generation_max_length=MAX_LEN, per_device_eval_batch_size=1,
    fp16=False, bf16=False, report_to='none',
)
collator_nllb  = DataCollatorForSeq2Seq(tok_nllb, model=model_nllb, pad_to_multiple_of=8)
trainer_nllb_r = Seq2SeqTrainer(
    model=model_nllb, args=args_nllb,
    tokenizer=tok_nllb, data_collator=collator_nllb,
    compute_metrics=compute_metrics_fn(tok_nllb),
)
print("Evaluating NLLB...")
eval_nllb = trainer_nllb_r.evaluate(eval_dataset=tok_dev_nllb)
print("Predicting NLLB...")
pred_nllb = trainer_nllb_r.predict(tok_test_nllb)
print("NLLB done:", eval_nllb)

print("\n✅ All variables ready! Now run your TABLE 3/4 cell.")

In [ ]:
# @title
import time
import torch
from transformers import (
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    AutoModelForSeq2SeqLM, NllbTokenizerFast,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
import evaluate
import numpy as np

# ── Paths ─────────────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/model_outputs'

# ── Helper functions ──────────────────────────────────────────────────────
def fmt_time(seconds):
    m, s = divmod(int(seconds), 60)
    return f"{m}m {s}s"

def free_memory():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Metrics ───────────────────────────────────────────────────────────────
bleu_metric = evaluate.load('sacrebleu')

def compute_metrics_fn(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds  = [p.strip() for p in decoded_preds]
        decoded_labels = [[l.strip()] for l in decoded_labels]
        result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
        return {'bleu': round(result['score'], 4)}
    return compute_metrics

print("Imports done!")

In [ ]:
# @title
import time
import torch

# ── Recover training times from logs ─────────────────────────────────────
m2m_train_time  = 63 * 60 + 57   # 63m 57s  (from your M2M100 output)
nllb_train_time = 90 * 60 + 0    # 90m 00s  (from your NLLB output)

# ── Reload M2M100 from best checkpoint ───────────────────────────────────
print("Reloading M2M100 from checkpoint...")
tok_m2m_r   = M2M100Tokenizer.from_pretrained(f'{OUTPUT_DIR}/m2m100_okun/checkpoint-490',
                                               src_lang='en', tgt_lang='yo')
model_m2m_r = M2M100ForConditionalGeneration.from_pretrained(
                  f'{OUTPUT_DIR}/m2m100_okun/checkpoint-490', torch_dtype=torch.float32)
model_m2m_r.config.forced_bos_token_id = tok_m2m_r.get_lang_id('yo')

trainer_m2m_r = Seq2SeqTrainer(
    model           = model_m2m_r,
    args            = args_m2m,          # reuse your existing args
    eval_dataset    = tok_dev_m2m,
    tokenizer       = tok_m2m_r,
    data_collator   = collator_m2m,
    compute_metrics = compute_metrics_fn(tok_m2m_r),
)
print("Evaluating M2M100...")
eval_m2m = trainer_m2m_r.evaluate(eval_dataset=tok_dev_m2m)
print("Predicting M2M100...")
pred_m2m = trainer_m2m_r.predict(tok_test_m2m)
print("M2M100 done:", eval_m2m)

# ── Reload NLLB from best checkpoint ─────────────────────────────────────
print("\nReloading NLLB from checkpoint...")
tok_nllb_r   = NllbTokenizerFast.from_pretrained(f'{OUTPUT_DIR}/nllb_okun/checkpoint-392',
                                                  src_lang='eng_Latn', tgt_lang='yor_Latn')
model_nllb_r = AutoModelForSeq2SeqLM.from_pretrained(
                   f'{OUTPUT_DIR}/nllb_okun/checkpoint-392', torch_dtype=torch.float32)
model_nllb_r.config.forced_bos_token_id = tok_nllb_r.convert_tokens_to_ids('yor_Latn')

trainer_nllb_r = Seq2SeqTrainer(
    model           = model_nllb_r,
    args            = args_nllb,         # reuse your existing args
    eval_dataset    = tok_dev_nllb,
    tokenizer       = tok_nllb_r,
    data_collator   = collator_nllb,
    compute_metrics = compute_metrics_fn(tok_nllb_r),
)
print("Evaluating NLLB...")
eval_nllb = trainer_nllb_r.evaluate(eval_dataset=tok_dev_nllb)
print("Predicting NLLB...")
pred_nllb = trainer_nllb_r.predict(tok_test_nllb)
print("NLLB done:", eval_nllb)

print("\nAll variables restored! Now run your TABLE 3/4 cell.")

In [ ]:
import json
with open(f'{OUTPUT_DIR}/results.json', 'r') as f:
    saved = json.load(f)
print(json.dumps(saved, indent=2))

In [ ]:
# @title
from transformers import AutoModelForSeq2SeqLM, NllbTokenizerFast

NLLB_MODEL  = 'facebook/nllb-200-distilled-600M'
NLLB_OUTDIR = f'{OUTPUT_DIR}/nllb_okun'
NLLB_SRC    = 'eng_Latn'
NLLB_TGT    = 'yor_Latn'   # Yoruba — closest available proxy for Okun

print('Loading NLLB-200 tokenizer and model...')
tok_nllb = NllbTokenizerFast.from_pretrained(
    NLLB_MODEL, src_lang=NLLB_SRC, tgt_lang=NLLB_TGT
)
forced_bos_nllb = tok_nllb.convert_tokens_to_ids(NLLB_TGT)

model_nllb = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
model_nllb.config.forced_bos_token_id = forced_bos_nllb



def preprocess_nllb(batch):
    tok_nllb.src_lang = NLLB_SRC
    model_inputs = tok_nllb(
        batch['english'],
        max_length=MAX_LEN, truncation=True, padding='max_length'
    )
    with tok_nllb.as_target_tokenizer():
        labels = tok_nllb(
            batch['okun'],
            max_length=MAX_LEN, truncation=True, padding='max_length'
        )
    label_ids = labels['input_ids']
    label_ids = [
        [(t if t != tok_nllb.pad_token_id else -100) for t in l]
        for l in label_ids
    ]
    model_inputs['labels'] = label_ids
    return model_inputs

print('Tokenising...')
tok_train_nllb = train_ds.map(preprocess_nllb, batched=True, remove_columns=train_ds.column_names)
tok_dev_nllb   = dev_ds.map(preprocess_nllb,   batched=True, remove_columns=dev_ds.column_names)
tok_test_nllb  = test_ds.map(preprocess_nllb,  batched=True, remove_columns=test_ds.column_names)

args_nllb = Seq2SeqTrainingArguments(
    output_dir                  = NLLB_OUTDIR,
    num_train_epochs            = 10,
    per_device_train_batch_size = 2,       # NLLB is larger; reduce batch size
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 16,       # effective batch = 32
    learning_rate               = 5e-4,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    fp16                        = False,
    bf16                        = True,
    predict_with_generate       = True,
    generation_max_length       = MAX_LEN,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    greater_is_better           = True,
    save_total_limit            = 2,
    dataloader_num_workers      = 2,
    logging_steps               = 20,
    report_to                   = 'none',
)

collator_nllb = DataCollatorForSeq2Seq(tok_nllb, model=model_nllb, pad_to_multiple_of=8)

trainer_nllb = Seq2SeqTrainer(
    model           = model_nllb,
    args            = args_nllb,
    train_dataset   = tok_train_nllb,
    eval_dataset    = tok_dev_nllb,
    tokenizer       = tok_nllb,
    data_collator   = collator_nllb,
    compute_metrics = compute_metrics_fn(tok_nllb),
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print('\nStarting NLLB-200 training...')
t0 = time.time()
trainer_nllb.train()
nllb_train_time = time.time() - t0
print(f'\nNLLB-200 training done in {fmt_time(nllb_train_time)}')

print('Evaluating on dev set...')
eval_nllb = trainer_nllb.evaluate(eval_dataset=tok_dev_nllb)

print('Predicting on test set...')
pred_nllb = trainer_nllb.predict(tok_test_nllb)

print('\nNLLB-200 done.')
free_memory()

## Step 9 — Results summary
Copy these numbers directly into Tables 3 and 4 of your paper.

In [ ]:
import os
print(os.listdir(f'{OUTPUT_DIR}'))

In [ ]:
import json
import os

# Check what's available
print("Files in output dir:", os.listdir(OUTPUT_DIR))

# Load results if file exists
if os.path.exists(f'{OUTPUT_DIR}/results.json'):
    with open(f'{OUTPUT_DIR}/results.json', 'r') as f:
        saved = json.load(f)
    eval_m2m = saved.get('eval_m2m', {})
    eval_nllb = saved.get('eval_nllb', {})
    print("Loaded successfully!")
else:
    print("results.json not found — run the evaluation cell first")

In [ ]:
import os

print("M2M file size:", os.path.getsize(f"{OUTPUT_DIR}/m2m_eval.json"))
print("NLLB file size:", os.path.getsize(f"{OUTPUT_DIR}/nllb_eval.json"))

In [ ]:
import json
import os

# ── 1. Helper functions ──────────────────────────────────────────────────
def fmt_time(seconds):
    m, s = divmod(int(seconds), 60)
    return f"{m}m {s}s"

def get(d, *keys):
    for k in keys:
        if k in d:
            return round(d[k], 4)
    return 'N/A'

# ── 2. Load saved results ────────────────────────────────────────────────
m2m_eval_path = f"{OUTPUT_DIR}/m2m_eval.json"
nllb_eval_path = f"{OUTPUT_DIR}/nllb_eval.json"
results_path = f"{OUTPUT_DIR}/results.json"

# fallback values
m2m_train_time = 63 * 60 + 57
nllb_train_time = 90 * 60 + 0

eval_m2m = {}
eval_nllb = {}
pred_m2m_metrics = {}
pred_nllb_metrics = {}

# Load eval files
if os.path.exists(m2m_eval_path):
    with open(m2m_eval_path) as f:
        eval_m2m = json.load(f)

if os.path.exists(nllb_eval_path):
    with open(nllb_eval_path) as f:
        eval_nllb = json.load(f)

# Load combined results if available
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)

    pred_m2m_metrics = results.get("M2M100", {}).get("predict", {})
    pred_nllb_metrics = results.get("NLLB200", {}).get("predict", {})
    m2m_train_time = results.get("M2M100", {}).get("train_time_seconds", m2m_train_time)
    nllb_train_time = results.get("NLLB200", {}).get("train_time_seconds", nllb_train_time)

# ── 3. Print clean tables ────────────────────────────────────────────────
sep = '='*55
print(sep)
print('  COPY THESE INTO YOUR PAPER')
print(sep)
print()

# ================= M2M =================
print('TABLE 3 — Evaluation metrics (dev set)')
print(f'  Eval BLEU   — M2M100 : {get(eval_m2m, "eval_bleu")}')
print(f'  Eval Loss   — M2M100 : {get(eval_m2m, "eval_loss")}')
print(f'  Eval GenLen — M2M100 : {get(eval_m2m, "eval_gen_len")}')
print(f'  Eval Runtime— M2M100 : {fmt_time(eval_m2m.get("eval_runtime", 0))}')
print()

print('TABLE 4 — Prediction metrics (test set)')
print(f'  Pred BLEU   — M2M100 : {get(pred_m2m_metrics, "test_bleu")}')
print(f'  Pred Loss   — M2M100 : {get(pred_m2m_metrics, "test_loss")}')
print(f'  Pred GenLen — M2M100 : {get(pred_m2m_metrics, "test_gen_len")}')
print(f'  Pred Runtime— M2M100 : {fmt_time(pred_m2m_metrics.get("test_runtime", 0))}')
print()

print('TRAINING TIMES')
print(f'  M2M100 : {fmt_time(m2m_train_time)}')
print(sep)

# ================= NLLB =================
print('TABLE 3 — Evaluation metrics (dev set)')
print(f'  Eval BLEU   — NLLB   : {get(eval_nllb, "eval_bleu")}')
print(f'  Eval Loss   — NLLB   : {get(eval_nllb, "eval_loss")}')
print(f'  Eval GenLen — NLLB   : {get(eval_nllb, "eval_gen_len")}')
print(f'  Eval Runtime— NLLB   : {fmt_time(eval_nllb.get("eval_runtime", 0))}')
print()

print('TABLE 4 — Prediction metrics (test set)')
print(f'  Pred BLEU   — NLLB   : {get(pred_nllb_metrics, "test_bleu")}')
print(f'  Pred Loss   — NLLB   : {get(pred_nllb_metrics, "test_loss")}')
print(f'  Pred GenLen — NLLB   : {get(pred_nllb_metrics, "test_gen_len")}')
print(f'  Pred Runtime— NLLB   : {fmt_time(pred_nllb_metrics.get("test_runtime", 0))}')
print()

print('TRAINING TIMES')
print(f'  NLLB   : {fmt_time(nllb_train_time)}')
print(sep)

In [ ]:
def get(d, *keys):
    for k in keys:
        if k in d: return round(d[k], 4)
    return 'N/A'

sep = '='*55
print(sep)
print('  COPY THESE INTO YOUR PAPER')
print(sep)
print()
print('TABLE 3 — Evaluation metrics (dev set)')
print(f'  Eval BLEU   — M2M100 : {get(eval_m2m, "eval_bleu")}')
print(f'  Eval Loss   — M2M100 : {get(eval_m2m, "eval_loss")}')
print(f'  Eval GenLen — M2M100 : {get(eval_m2m, "eval_gen_len")}')
print(f'  Eval Runtime— M2M100 : {fmt_time(eval_m2m.get("eval_runtime", 0))}')
print()
print('TABLE 4 — Prediction metrics (test set)')
print(f'  Pred BLEU   — M2M100 : {get(pred_m2m.metrics, "test_bleu")}')
print(f'  Pred Loss   — M2M100 : {get(pred_m2m.metrics, "test_loss")}')
print(f'  Pred GenLen — M2M100 : {get(pred_m2m.metrics, "test_gen_len")}')
print(f'  Pred Runtime— M2M100 : {fmt_time(pred_m2m.metrics.get("test_runtime", 0))}')
print()
print('TRAINING TIMES')
print(f'  M2M100 : {fmt_time(m2m_train_time)}')
print(sep)

print('TABLE 3 — Evaluation metrics (dev set)')
print(f'  Eval BLEU   — NLLB   : {get(eval_nllb, "eval_bleu")}')
print(f'  Eval Loss   — NLLB   : {get(eval_nllb, "eval_loss")}')
print(f'  Eval GenLen — NLLB   : {get(eval_nllb, "eval_gen_len")}')
print(f'  Eval Runtime— NLLB   : {fmt_time(eval_nllb.get("eval_runtime", 0))}')
print()
print('TABLE 4 — Prediction metrics (test set)')
print(f'  Pred BLEU   — NLLB   : {get(pred_nllb.metrics, "test_bleu")}')
print(f'  Pred Loss   — NLLB   : {get(pred_nllb.metrics, "test_loss")}')
print(f'  Pred GenLen — NLLB   : {get(pred_nllb.metrics, "test_gen_len")}')
print(f'  Pred Runtime— NLLB   : {fmt_time(pred_nllb.metrics.get("test_runtime", 0))}')
print()
print('TRAINING TIMES')
print(f'  NLLB   : {fmt_time(nllb_train_time)}')
print(sep)
# Also save to Drive
import json
results = {
    'M2M100': {'eval': dict(eval_m2m), 'predict': dict(pred_m2m.metrics),
               'train_time_seconds': m2m_train_time},
    'NLLB200': {'eval': dict(eval_nllb), 'predict': dict(pred_nllb.metrics),
                'train_time_seconds': nllb_train_time},
}
results_path = f'{OUTPUT_DIR}/results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults also saved to: {results_path}')

In [ ]:
def get(d, *keys):
    for k in keys:
        if k in d: return round(d[k], 4)
    return 'N/A'

sep = '='*55
print(sep)
print('  COPY THESE INTO YOUR PAPER')
print(sep)
print()
print('TABLE 3 — Evaluation metrics (dev set)')
print(f'  Eval BLEU   — M2M100 : {get(eval_m2m, "eval_bleu")}')
print(f'  Eval BLEU   — NLLB   : {get(eval_nllb, "eval_bleu")}')
print(f'  Eval Loss   — M2M100 : {get(eval_m2m, "eval_loss")}')
print(f'  Eval Loss   — NLLB   : {get(eval_nllb, "eval_loss")}')
print(f'  Eval GenLen — M2M100 : {get(eval_m2m, "eval_gen_len")}')
print(f'  Eval GenLen — NLLB   : {get(eval_nllb, "eval_gen_len")}')
print(f'  Eval Runtime— M2M100 : {fmt_time(eval_m2m.get("eval_runtime", 0))}')
print(f'  Eval Runtime— NLLB   : {fmt_time(eval_nllb.get("eval_runtime", 0))}')
print()
print('TABLE 4 — Prediction metrics (test set)')
print(f'  Pred BLEU   — M2M100 : {get(pred_m2m.metrics, "test_bleu")}')
print(f'  Pred BLEU   — NLLB   : {get(pred_nllb.metrics, "test_bleu")}')
print(f'  Pred Loss   — M2M100 : {get(pred_m2m.metrics, "test_loss")}')
print(f'  Pred Loss   — NLLB   : {get(pred_nllb.metrics, "test_loss")}')
print(f'  Pred GenLen — M2M100 : {get(pred_m2m.metrics, "test_gen_len")}')
print(f'  Pred GenLen — NLLB   : {get(pred_nllb.metrics, "test_gen_len")}')
print(f'  Pred Runtime— M2M100 : {fmt_time(pred_m2m.metrics.get("test_runtime", 0))}')
print(f'  Pred Runtime— NLLB   : {fmt_time(pred_nllb.metrics.get("test_runtime", 0))}')
print()
print('TRAINING TIMES')
print(f'  M2M100 : {fmt_time(m2m_train_time)}')
print(f'  NLLB   : {fmt_time(nllb_train_time)}')
print(sep)

# Also save to Drive
import json
results = {
    'M2M100': {'eval': dict(eval_m2m), 'predict': dict(pred_m2m.metrics),
               'train_time_seconds': m2m_train_time},
    'NLLB200': {'eval': dict(eval_nllb), 'predict': dict(pred_nllb.metrics),
                'train_time_seconds': nllb_train_time},
}
results_path = f'{OUTPUT_DIR}/results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults also saved to: {results_path}')

## Step 9b — chr F++ Evaluation
Computes **chrF++** (character n-gram F-score with word bigrams, `word_order=2`) on both
the dev and test sets for M2M100 and NLLB-200.
chrF++ is reported alongside BLEU in Tables 5 and 6 of the paper because it is
sensitive to morphological and diacritical variation — the primary quality concern for Okun.

**Runs independently of the training session** — it reloads checkpoints from Drive
if `pred_m2m` / `pred_nllb` are not already in memory.

In [ ]:
M2M_CKPT = f'{OUTPUT_DIR}/m2m100_okun/checkpoint-392'

In [ ]:
# ── Step 9b: chrF++ evaluation ────────────────────────────────────────────
# Works in two modes:
#   (A) predictions already in memory from training session → decode + score
#   (B) new session / checkpoint reload          → reload model, re-predict

import numpy as np, json, os, torch
from sacrebleu.metrics import CHRF

chrf_metric = CHRF(word_order=2)   # chrF++

def decode_predictions(predictions, label_ids, tokenizer):
    """Decode token-ID arrays from trainer.predict() into string lists."""
    # Predictions: pad tokens → skip via skip_special_tokens
    # Labels:      -100      → replace with pad_token_id before decode
    preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)
    refs   = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return [p.strip() for p in preds], [r.strip() for r in refs]


def compute_chrf(predictions, label_ids, tokenizer):
    hyps, refs = decode_predictions(predictions, label_ids, tokenizer)
    return round(chrf_metric.corpus_score(hyps, [refs]).score, 2)


# ── Path constants (must match Steps 4 and 7/8) ───────────────────────────
OUTPUT_DIR  = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/model_outputs'
CORPUS_PATH = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/Okun Dataset Submission/finetune sample set.xlsx'
M2M_CKPT    = f'{OUTPUT_DIR}/m2m100_okun/checkpoint-392'
NLLB_CKPT   = f'{OUTPUT_DIR}/nllb_okun/checkpoint-440'
NLLB_SRC    = 'eng_Latn'
NLLB_TGT    = 'yor_Latn'
MAX_LEN     = 128


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ── Decide whether to reload or reuse in-memory predictions ───────────────
def need_reload(var_name):
    try:
        v = eval(var_name)
        return v is None or not hasattr(v, 'predictions')
    except NameError:
        return True


# ── Helper: reload tokenised datasets from corpus ─────────────────────────
def reload_corpus_splits():
    import pandas as pd
    from datasets import Dataset
    df = pd.read_excel(CORPUS_PATH)
    en_col, ok_col = df.columns[1], df.columns[2]
    filled = df[
        df[ok_col].notna() &
        (df[ok_col].astype(str).str.strip() != '') &
        (~df[ok_col].astype(str).str.contains(
            r'\[MISSING\]|\[UNVERIFIED\]|\[continues', na=False))
    ].copy()
    filled = filled.rename(columns={df.columns[0]:'reference', en_col:'english', ok_col:'okun'})
    filled['english'] = filled['english'].str.replace(r'\s+', ' ', regex=True).str.strip()
    filled['okun']    = filled['okun'].str.replace(r'\s+', ' ', regex=True).str.strip()
    n = len(filled)
    t_end, d_end = int(n*0.8), int(n*0.9)
    dev_ds  = Dataset.from_pandas(filled.iloc[t_end:d_end][['english','okun']].reset_index(drop=True))
    test_ds = Dataset.from_pandas(filled.iloc[d_end:][['english','okun']].reset_index(drop=True))
    return dev_ds, test_ds


# ── Score helper: tokenise a dataset with a given model and predict ────────
def predict_from_checkpoint(model, tokenizer, dataset, preprocess_fn, collator_cls):
    from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
    tok_ds = dataset.map(preprocess_fn, batched=True, remove_columns=dataset.column_names)
    args = Seq2SeqTrainingArguments(
        output_dir='/tmp/eval_tmp',
        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        per_device_eval_batch_size=8,
        report_to='none',
        fp16=False,
    )
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)
    trainer  = Seq2SeqTrainer(model=model, args=args, data_collator=collator)
    trainer.tokenizer = tokenizer
    return trainer.predict(tok_ds)


# ══════════════════════════════════════════════════════════════════════════
# M2M100
# ══════════════════════════════════════════════════════════════════════════
if need_reload('pred_m2m') or need_reload('tok_m2m'):
    print('Reloading M2M100 from checkpoint for chrF++ evaluation...')
    from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
    tok_m2m   = M2M100Tokenizer.from_pretrained(M2M_CKPT, src_lang='en', tgt_lang='yo')
    model_m2m = M2M100ForConditionalGeneration.from_pretrained(M2M_CKPT).to(device)
    model_m2m.generation_config.forced_bos_token_id = tok_m2m.get_lang_id('yo')
    dev_ds, test_ds = reload_corpus_splits()

    def preprocess_m2m_eval(batch):
        tok_m2m.src_lang = 'en'
        inp = tok_m2m(batch['english'], max_length=MAX_LEN, truncation=True, padding='max_length')
        lbl = tok_m2m(text_target=batch['okun'], max_length=MAX_LEN, truncation=True, padding='max_length')
        inp['labels'] = [[(t if t != tok_m2m.pad_token_id else -100) for t in l]
                         for l in lbl['input_ids']]
        return inp

    pred_m2m_dev  = predict_from_checkpoint(model_m2m, tok_m2m, dev_ds,  preprocess_m2m_eval, None)
    pred_m2m_test = predict_from_checkpoint(model_m2m, tok_m2m, test_ds, preprocess_m2m_eval, None)
else:
    print('Using in-memory M2M100 predictions...')
    # re-run predict on dev using the already-loaded trainer if available
    try:
        pred_m2m_dev  = trainer_m2m.predict(tok_dev_m2m)
    except NameError:
        dev_ds, test_ds = reload_corpus_splits()
        def preprocess_m2m_eval(batch):
            tok_m2m.src_lang = 'en'
            inp = tok_m2m(batch['english'], max_length=MAX_LEN, truncation=True, padding='max_length')
            lbl = tok_m2m(text_target=batch['okun'], max_length=MAX_LEN, truncation=True, padding='max_length')
            inp['labels'] = [[(t if t != tok_m2m.pad_token_id else -100) for t in l]
                             for l in lbl['input_ids']]
            return inp
        pred_m2m_dev = predict_from_checkpoint(model_m2m, tok_m2m, dev_ds, preprocess_m2m_eval, None)
    pred_m2m_test = pred_m2m   # from training session

m2m_dev_chrf  = compute_chrf(pred_m2m_dev.predictions,  pred_m2m_dev.label_ids,  tok_m2m)
m2m_test_chrf = compute_chrf(pred_m2m_test.predictions, pred_m2m_test.label_ids, tok_m2m)
print(f'M2M100  — Dev chrF++: {m2m_dev_chrf}  |  Test chrF++: {m2m_test_chrf}')


# ══════════════════════════════════════════════════════════════════════════
# NLLB-200
# ══════════════════════════════════════════════════════════════════════════
if need_reload('pred_nllb') or need_reload('tok_nllb'):
    print('Reloading NLLB-200 from checkpoint for chrF++ evaluation...')
    from transformers import AutoModelForSeq2SeqLM, NllbTokenizerFast
    tok_nllb   = NllbTokenizerFast.from_pretrained(
        'facebook/nllb-200-distilled-600M', src_lang=NLLB_SRC, tgt_lang=NLLB_TGT)
    model_nllb = AutoModelForSeq2SeqLM.from_pretrained(NLLB_CKPT).to(device)
    forced_bos_nllb = tok_nllb.convert_tokens_to_ids(NLLB_TGT)
    model_nllb.generation_config.forced_bos_token_id = forced_bos_nllb
    if 'dev_ds' not in dir():
        dev_ds, test_ds = reload_corpus_splits()

    def preprocess_nllb_eval(batch):
        tok_nllb.src_lang = NLLB_SRC
        inp = tok_nllb(batch['english'], max_length=MAX_LEN, truncation=True, padding='max_length')
        lbl = tok_nllb(text_target=batch['okun'], max_length=MAX_LEN, truncation=True, padding='max_length')
        inp['labels'] = [[(t if t != tok_nllb.pad_token_id else -100) for t in l]
                         for l in lbl['input_ids']]
        return inp

    pred_nllb_dev  = predict_from_checkpoint(model_nllb, tok_nllb, dev_ds,  preprocess_nllb_eval, None)
    pred_nllb_test = predict_from_checkpoint(model_nllb, tok_nllb, test_ds, preprocess_nllb_eval, None)
else:
    print('Using in-memory NLLB-200 predictions...')
    try:
        pred_nllb_dev = trainer_nllb.predict(tok_dev_nllb)
    except NameError:
        if 'dev_ds' not in dir():
            dev_ds, test_ds = reload_corpus_splits()
        def preprocess_nllb_eval(batch):
            tok_nllb.src_lang = NLLB_SRC
            inp = tok_nllb(batch['english'], max_length=MAX_LEN, truncation=True, padding='max_length')
            lbl = tok_nllb(text_target=batch['okun'], max_length=MAX_LEN, truncation=True, padding='max_length')
            inp['labels'] = [[(t if t != tok_nllb.pad_token_id else -100) for t in l]
                             for l in lbl['input_ids']]
            return inp
        pred_nllb_dev = predict_from_checkpoint(model_nllb, tok_nllb, dev_ds, preprocess_nllb_eval, None)
    pred_nllb_test = pred_nllb

nllb_dev_chrf  = compute_chrf(pred_nllb_dev.predictions,  pred_nllb_dev.label_ids,  tok_nllb)
nllb_test_chrf = compute_chrf(pred_nllb_test.predictions, pred_nllb_test.label_ids, tok_nllb)
print(f'NLLB-200 — Dev chrF++: {nllb_dev_chrf}  |  Test chrF++: {nllb_test_chrf}')


# ── Paper-ready summary ───────────────────────────────────────────────────
sep = '=' * 58
print(f'\n{sep}')
print('  COPY THESE INTO TABLES 5 AND 6 OF YOUR PAPER')
print(sep)
print()
print('TABLE 5 — Dev set')
print(f'  Eval chrF++  — M2M100  : {m2m_dev_chrf}')
print(f'  Eval chrF++  — NLLB-200: {nllb_dev_chrf}')
print()
print('TABLE 6 — Test set')
print(f'  Test chrF++  — M2M100  : {m2m_test_chrf}')
print(f'  Test chrF++  — NLLB-200: {nllb_test_chrf}')
print(sep)


# ── Append chrF++ to results.json ─────────────────────────────────────────
results_path = f'{OUTPUT_DIR}/results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
else:
    results = {}

results.setdefault('M2M100', {})['chrf']  = {'dev': m2m_dev_chrf,  'test': m2m_test_chrf}
results.setdefault('NLLB200', {})['chrf'] = {'dev': nllb_dev_chrf, 'test': nllb_test_chrf}

with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nchrF++ scores appended to: {results_path}')


## Step 10 — Sample translations (qualitative check)

In [ ]:
# Step 10 — Sample translations (qualitative check)
# Loads models fresh from saved checkpoints — works independently of training session

import torch, os
from transformers import (
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    AutoModelForSeq2SeqLM, NllbTokenizerFast
)
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Paths ─────────────────────────────────────────────────────────────────
OUTPUT_DIR  = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/model_outputs'
CORPUS_PATH = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/Okun Dataset Submission/finetune sample set.xlsx'
M2M_CKPT    = f'{OUTPUT_DIR}/m2m100_okun/checkpoint-392'
NLLB_CKPT   = f'{OUTPUT_DIR}/nllb_okun/checkpoint-440'
NLLB_SRC    = 'eng_Latn'
NLLB_TGT    = 'yor_Latn'
MAX_LEN     = 128

# ── Load corpus and split ─────────────────────────────────────────────────
import pandas as pd
df = pd.read_excel(CORPUS_PATH)
en_col, ok_col = df.columns[1], df.columns[2]
filled = df[
    df[ok_col].notna() &
    (df[ok_col].astype(str).str.strip() != '') &
    (~df[ok_col].astype(str).str.contains(r'\[MISSING\]|\[UNVERIFIED\]|\[continues', na=False))
].copy()
filled.columns = ['reference', 'english', 'okun'] if len(filled.columns) == 3 else filled.columns
filled = filled.rename(columns={df.columns[0]: 'reference', en_col: 'english', ok_col: 'okun'})
filled['english'] = filled['english'].str.replace(r'\s+', ' ', regex=True).str.strip()
filled['okun']    = filled['okun'].str.replace(r'\s+', ' ', regex=True).str.strip()

_, temp_df = train_test_split(filled, test_size=0.2, random_state=42)
_, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
test_df = test_df.reset_index(drop=True)
print(f'Test set: {len(test_df)} sentences')

# ── Load M2M100 from checkpoint ───────────────────────────────────────────
print('\nLoading M2M100 from checkpoint...')
tok_m2m = M2M100Tokenizer.from_pretrained(M2M_CKPT, src_lang='en', tgt_lang='yo')
forced_bos = tok_m2m.get_lang_id('yo')
model_m2m = M2M100ForConditionalGeneration.from_pretrained(M2M_CKPT).float().to(device)
model_m2m.eval()

# ── Load NLLB from checkpoint ─────────────────────────────────────────────
print('Loading NLLB-200 from checkpoint...')
tok_nllb = NllbTokenizerFast.from_pretrained('facebook/nllb-200-distilled-600M',
                                              src_lang=NLLB_SRC, tgt_lang=NLLB_TGT)
forced_bos_nllb = tok_nllb.convert_tokens_to_ids(NLLB_TGT)
model_nllb = AutoModelForSeq2SeqLM.from_pretrained(NLLB_CKPT).float().to(device)
model_nllb.generation_config.forced_bos_token_id = forced_bos_nllb
model_nllb.eval()

# ── Generate sample translations ──────────────────────────────────────────
print('\nSample translations from each model on the test set:\n')

sample_indices = [0, 5, 10]
sample_en  = [test_df['english'].iloc[i] for i in sample_indices]
sample_ref = [test_df['okun'].iloc[i]    for i in sample_indices]

# M2M100
tok_m2m.src_lang = 'en'
with torch.no_grad():
    inputs_m2m = tok_m2m(sample_en, return_tensors='pt', padding=True,
                          truncation=True, max_length=MAX_LEN).to(device)
    out_m2m = model_m2m.generate(**inputs_m2m, forced_bos_token_id=forced_bos,
                                   max_length=MAX_LEN, num_beams=4)
trans_m2m = tok_m2m.batch_decode(out_m2m, skip_special_tokens=True)

# NLLB
tok_nllb.src_lang = NLLB_SRC
with torch.no_grad():
    inputs_nllb = tok_nllb(sample_en, return_tensors='pt', padding=True,
                             truncation=True, max_length=MAX_LEN).to(device)
    out_nllb = model_nllb.generate(**inputs_nllb, forced_bos_token_id=forced_bos_nllb,
                                     max_length=MAX_LEN, num_beams=4)
trans_nllb = tok_nllb.batch_decode(out_nllb, skip_special_tokens=True)

for i, (en, ref, m2m, nllb) in enumerate(zip(sample_en, sample_ref, trans_m2m, trans_nllb), 1):
    print(f'--- Sample {i} ---')
    print(f'English   : {en}')
    print(f'Reference : {ref}')
    print(f'M2M100    : {m2m}')
    print(f'NLLB-200  : {nllb}')
    print()
